# AI 201 — Week 1 Demo: Chunking Strategy Comparison

**Demo question:** Does chunking strategy actually change what gets retrieved — and does it change the answer?

Today we're going to:
1. **Write** a fixed-size chunker from scratch
2. **Compare** it against two more sophisticated strategies (already built)
3. **Write** a RAG pipeline that retrieves chunks and sends them to an LLM
4. **Run** the same query through all three strategies and compare what comes back

---
**Before running:** `python setup/build_indexes.py` must have been run first.

In [ ]:
# Cell 1 — Setup (run this first, then minimize it)
from dotenv import load_dotenv
load_dotenv()

import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from utils.vector_store import VectorStore
from utils.llm import LLMClient
from utils.evaluation import evaluate_results, print_comparison_table, format_result_for_display
from chunkers import SemanticChunker, RecursiveChunker

store = VectorStore(persist_dir='./chroma_db')
llm = LLMClient()
STRATEGIES = ['fixed_size', 'semantic', 'recursive']

# Load the Marineford page — we'll chunk this live
page_text = Path('data/wiki_pages/marineford_arc.txt').read_text()

print('Ready.')
print(f'Wiki page loaded: {len(page_text):,} characters')
print(f'LLM model: {llm.model}')

---
## Part 1: Write a Chunker

Let's start from scratch. A chunker is just a function that takes a string and returns a list of smaller strings.

The simplest possible strategy: split every N characters, with a little overlap so we don't cut context cold.

**✍️ LIVE CODE — write this together:**

In [ ]:
# ── INSTRUCTOR: Write this live. Talk through each line as you type. ────────
#
# Goal: a function that splits text into fixed-size chunks with overlap.
# Ask students: "What parameters do we need?" (chunk_size, overlap)
# Ask students: "How do we know when to stop?" (while start < len(text))
#
# Target: ~10 lines. Should take about 3 minutes.
# ────────────────────────────────────────────────────────────────────────────

def chunk_fixed(text, chunk_size=500, overlap=50):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap  # slide forward, keeping overlap
    return chunks


# Try it on the Marineford page
fixed_chunks = chunk_fixed(page_text, chunk_size=500, overlap=50)
print(f'Fixed-size produced {len(fixed_chunks)} chunks')
print(f'Average chunk size: {sum(len(c) for c in fixed_chunks) // len(fixed_chunks)} chars')

In [ ]:
# ── DEMO MOMENT 1 ──────────────────────────────────────────────────────────
# Look at a chunk that cuts mid-sentence — this is the problem.
# Ask students: "Is this a complete thought? What's missing?"
# ───────────────────────────────────────────────────────────────────────────

print('=== What fixed-size chunking looks like at a boundary ===')
print()

# Find a chunk near the Ace death section
ace_idx = next((i for i, c in enumerate(fixed_chunks) if 'magma fist' in c or 'Ace dies' in c), 8)

for i in [ace_idx - 1, ace_idx]:
    print(f'--- Chunk {i} ({len(fixed_chunks[i])} chars) ---')
    print(fixed_chunks[i])
    print()

print('↑ Notice where the chunk boundary landed. Did it cut mid-thought?')

---
### Now compare: semantic and recursive chunkers

These two strategies are already built — we'll look at what they produce on the same passage without writing them from scratch. The implementations are in `chunkers/semantic.py` and `chunkers/recursive.py` if you want to read them later.

**Semantic chunker:** groups sentences by meaning. When the topic shifts, it starts a new chunk.  
**Recursive chunker:** splits on a hierarchy of separators — paragraph breaks first, then sentences, then words.

In [ ]:
# Pre-built chunkers — compare output on the same page
semantic  = SemanticChunker(similarity_threshold=0.5, max_chunk_size=1000)
recursive = RecursiveChunker(chunk_size=600, overlap=60)

semantic_chunks  = semantic.chunk(page_text)
recursive_chunks = recursive.chunk(page_text)

print(f'Fixed-size:  {len(fixed_chunks)} chunks,   avg {sum(len(c) for c in fixed_chunks)//len(fixed_chunks)} chars')
print(f'Semantic:    {len(semantic_chunks)} chunks,   avg {sum(len(c["text"]) for c in semantic_chunks)//len(semantic_chunks)} chars')
print(f'Recursive:   {len(recursive_chunks)} chunks,  avg {sum(len(c["text"]) for c in recursive_chunks)//len(recursive_chunks)} chars')

In [ ]:
# ── DEMO MOMENT 2 ──────────────────────────────────────────────────────────
# Show how the semantic chunker handles the same passage.
# Ask: "Why did these sentences end up together? What changed the topic?"
# ───────────────────────────────────────────────────────────────────────────

print('=== Semantic chunk covering Ace\'s death ===')
print()
ace_s = [c for c in semantic_chunks if 'magma fist' in c['text'] or 'Ace dies' in c['text']]
for c in ace_s:
    print(f'--- Chunk {c["index"]} ({len(c["text"])} chars, {c["sentence_count"]} sentences) ---')
    print(c['text'])
    print()

print('=== Recursive chunk covering the same passage ===')
print()
ace_r = [c for c in recursive_chunks if 'magma fist' in c['text'] or 'Ace dies' in c['text']]
for c in ace_r:
    print(f'--- Chunk {c["index"]} ({len(c["text"])} chars) ---')
    print(c['text'])
    print()

---
### Under the Hood: What the VectorStore Is Actually Doing

The `VectorStore` wrapper handles ChromaDB calls for us in this demo. In the lab and project, you'll call ChromaDB directly — so let's see what's actually happening one level down.

In [ ]:
# ── DEMO MOMENT 3 ──────────────────────────────────────────────────────────
# Run the query and look at what each strategy retrieved.
# Ask: "Which top chunk gave the retriever the most complete picture?"
# Try swapping the query — suggestions at the bottom of the notebook.
# ───────────────────────────────────────────────────────────────────────────

QUERY = "What happened to Ace at Marineford?"

print(f'Query: "{QUERY}"\n')
results_by_strategy = {}

for strategy in STRATEGIES:
    results = store.query(strategy, QUERY, n_results=3)
    results_by_strategy[strategy] = results
    print(f'[{strategy}]  top score: {results[0]["score"]:.4f}')
    print(f'  {results[0]["text"][:220]}...')
    print()

# ── SIMILARITY SCORE INTERPRETATION ─────────────────────────────────────────
# Important: the number printed above is NOT the raw distance from ChromaDB.
# VectorStore.query() converts ChromaDB's L2 distance into a normalized 0–1
# similarity score (similarity = 1 - distance / 2), so a HIGHER score means
# the chunk is MORE similar to the query. 1.0 would be a perfect match.
#
#   ~0.6+      → strong match: the chunk directly addresses the query
#   ~0.4–0.6   → loosely related: some overlap, but may not answer the question
#   below ~0.4 → probably off-topic: the retriever is reaching
#
# (These bands are calibrated to this demo's embeddings — on the Ace/Marineford
# query the relevant chunks land around 0.62–0.69, which is a strong match here.)
#
# If your top scores are all low, the most likely cause is chunks that are too
# small to embed meaningfully — a sentence fragment doesn't carry enough
# semantic signal to match accurately. Inspect the chunks, not just the scores.
# ─────────────────────────────────────────────────────────────────────────────
print("Similarity score guide (higher = better): 0.6+ = strong match | 0.4–0.6 = loosely related | <0.4 = probably off-topic")

---
## Part 3: Write the RAG Pipeline

Retrieval gave us chunks. Now we need to send them to an LLM and get an answer.

This is the RAG pipeline in its simplest form: **retrieve → assemble context → call LLM**.

**✍️ LIVE CODE — write this together.**

One thing to nail before we start: the system prompt. Most RAG systems fail here quietly — not because of bad retrieval, but because the model ignores the retrieved context and answers from its training data anyway. We're going to write a weak version first, then fix it.

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# Goal: a function that takes a query + retrieved chunks and returns an answer.
#
# GROUNDING PROMPT — do this part first, as a class discussion:
#
#   Step 1: Write the weak version together. Type this and ask students:
#           "What's wrong with this instruction?"
#
#     system = "Answer questions about One Piece accurately using the context."
#
#   They should identify: "accurately" is vague — the model can decide what
#   counts as accurate. It has room to supplement from training data.
#
#   Step 2: Ask: "How would you rewrite it to close that gap?"
#   Take 2-3 answers from chat, then land on the strong version below.
#
#   Step 3: Note the source label in context assembly ([Source: ...]).
#   Ask: "Why did we include the source in the context string?"
#   (So we can instruct the model to cite it — and so users can verify.)
#
# Target: ~15 lines. Should take about 5 minutes including the discussion.
# ────────────────────────────────────────────────────────────────────────────

def rag_answer(query, chunks, llm_client):
    # Step 1: Assemble context — include source so the model can cite it
    context = "\n\n---\n\n".join(
        f"[Source: {c['source']}]\n{c['text']}"
        for c in chunks
    )

    # Step 2: Build the prompt
    #
    # WEAK (write this first, then cross it out):
    #   "Answer questions about One Piece accurately using the context."
    #
    # STRONG (revise to this):
    system = (
        "Answer the question using only the text provided below. "
        "Cite the source document for any claim you make (e.g., 'According to marineford_arc.txt...'). "
        "If the answer is not in the provided text, say so explicitly — "
        "do not draw on outside knowledge or fill in gaps from what you know about One Piece."
    )
    user = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    # Step 3: Call the LLM
    response = llm_client.client.chat.completions.create(
        model=llm_client.model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


# Quick test
test_answer = rag_answer(QUERY, results_by_strategy['fixed_size'], llm)
print(f'[fixed_size answer]\n{test_answer}')
print()
print('Check: does the response cite a source? Could this answer have come from')
print('anywhere other than the retrieved chunks? If yes — grounding failure.')

---
## Part 3: Write the RAG Pipeline

Retrieval gave us chunks. Now we need to send them to an LLM and get an answer.

This is the RAG pipeline in its simplest form: **retrieve → assemble context → call LLM**.

**✍️ LIVE CODE — write this together:**

In [ ]:
# ── INSTRUCTOR: Write this live. ────────────────────────────────────────────
#
# Goal: a function that takes a query + a list of retrieved chunks
# and returns an LLM answer.
#
# Ask students before writing:
#   "What do we need to send to the LLM?" (the question + the context)
#   "How do we format the context?" (join the chunks into one string)
#   "What should we tell the model in the system prompt?"
#     (answer from context only, be honest if you don't know)
#
# Target: ~15 lines. Should take about 4 minutes.
# ────────────────────────────────────────────────────────────────────────────

def rag_answer(query, chunks, llm_client):
    # Step 1: Assemble the context from retrieved chunks
    context = "\n\n---\n\n".join(
        f"[Source: {c['source']} | Score: {c['score']}]\n{c['text']}"
        for c in chunks
    )

    # Step 2: Build the prompt
    system = (
        "You are a helpful assistant answering questions about One Piece. "
        "Use only the provided context to answer. "
        "If the context doesn't have enough information, say so."
    )
    user = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"

    # Step 3: Call the LLM
    response = llm_client.client.chat.completions.create(
        model=llm_client.model,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=0.2,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


# Quick test with the fixed-size results
test_answer = rag_answer(QUERY, results_by_strategy['fixed_size'], llm)
print(f'[fixed_size answer]\n{test_answer}')

In [ ]:
# ── DEMO MOMENT 4 ──────────────────────────────────────────────────────────
# Run our pipeline for all three strategies. Compare the answers.
# Ask: "Why are these answers different if we used the exact same model?"
# (The model can only work with what retrieval gave it.)
# ───────────────────────────────────────────────────────────────────────────

print(f'Query: "{QUERY}"\n')
print('Generating answers for all three strategies...\n')

for strategy in STRATEGIES:
    answer = rag_answer(QUERY, results_by_strategy[strategy], llm)
    print(f'[{strategy.upper()}]')
    print(answer)
    print()

---
## Part 4: Measure It

"It gave a different answer" is a vibe. In production you need numbers.

Here's a simple comparison table using the retrieval similarity scores. In a production system you'd add faithfulness and answer relevance using a framework like RAGAS.

In [ ]:
# ── DEMO MOMENT 5 ──────────────────────────────────────────────────────────
# Evaluation table. Ask:
#   "What else would you measure beyond retrieval score?"
#   Guide toward: faithfulness (did it make things up?),
#   answer relevance (did it actually answer the question?),
#   groundedness (every claim supported by context?)
# ───────────────────────────────────────────────────────────────────────────

eval_output = evaluate_results(QUERY, results_by_strategy)
print_comparison_table(eval_output)
print(f"Strategy with highest avg retrieval score: '{eval_output['winner']}'")

---
## Try a Different Query

Change `QUERY` below and re-run cells 6–9 to see how results shift with a different question.

Good options:
- `"How does Haki counter Logia Devil Fruit powers?"`
- `"Why did Ace turn back to fight Akainu instead of escaping?"`
- `"What happened to Ace's Devil Fruit after he died?"`
- `"What was Whitebeard's reputation before Marineford?"`

In [ ]:
# ── TRY IT — take a query from a student ───────────────────────────────────

QUERY = "Why did Ace turn back to fight Akainu instead of escaping?"  # ← change me

custom_results = {s: store.query(s, QUERY, n_results=3) for s in STRATEGIES}

print(f'Query: "{QUERY}"\n')
for strategy in STRATEGIES:
    answer = rag_answer(QUERY, custom_results[strategy], llm)
    print(f'[{strategy.upper()}]')
    print(answer)
    print()

print_comparison_table(evaluate_results(QUERY, custom_results))